# NeurIPS 2026 replication, notebook 3: analysis and tables (GPU recommended)
Runs every head, statistic, fit and table of both papers from the cached features (notebooks 1 and 2 must have run first).

Two optimizer arms are run and kept side by side:
* `adamw`: the papers' printed head-training protocol (AdamW, lr 1e-3, wd 1e-4, cosine; 20 epochs for Paper A's TC-gap heads, 50 epochs per task for Paper B).
* `lbfgs`: the exact regularized minimizers (full-batch L-BFGS in float64), which is what the theorems are about.

On a T4 the two arms together take roughly 1.5 to 2.5 hours (the E_CF sweep up to K = 32,000 dominates); on the CPU runtime they take most of a day, so use a GPU runtime. Results go to Drive under `results/<optimizer>/` and `tables_out/<optimizer>/`; the run is restartable per arm.

In [ ]:
# ---- 0. Setup: Drive, repo, packages ---------------------------------------------
# Run this cell first in every notebook. Session state is lost between sessions;
# the feature caches on Drive are not.
from google.colab import drive; drive.mount('/content/drive')
import os, subprocess, sys, pathlib
WORK = pathlib.Path('/content/drive/MyDrive/neurips2026_replication'); WORK.mkdir(parents=True, exist_ok=True)
REPO_ZIP = WORK / 'repo.zip'            # upload the repo zip here once (claude_replication/repo.zip), or set GIT_URL
GIT_URL = ''                            # e.g. 'https://github.com/<owner>/<repo>.git' once the repo is public
if GIT_URL:
    subprocess.run(['git','clone','--depth','1',GIT_URL,'/content/repo'], check=True)
else:
    assert REPO_ZIP.exists(), f'upload the repo zip to {REPO_ZIP}'
    subprocess.run(['rm','-rf','/content/repo','/content/repo_unz']); subprocess.run(['unzip','-q','-o',str(REPO_ZIP),'-d','/content/repo_unz'], check=True)
    inner = next(p for p in pathlib.Path('/content/repo_unz').rglob('replicate.py')).parent
    subprocess.run(['mv', str(inner), '/content/repo'], check=True)
os.chdir('/content/repo'); sys.path.insert(0, '/content/repo')
os.environ['REPLICATION_FEATURES'] = str(WORK / 'features')
subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.44','datasets','gdown','scipy','scikit-learn'], check=True)
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('features ->', os.environ['REPLICATION_FEATURES'])

In [ ]:
PAPER = 'AB'                 # 'A', 'B' or 'AB'
ARMS = ['adamw', 'lbfgs']    # drop one to save time
SEEDS = '0,1,2,3,4'
RANPAC_M = 10000             # RanPAC projection width; 10000 as in RanPAC (needs the GPU; use 2000 on CPU)
SCHEDULES = ['printed', 'consistent']   # Paper A Algorithm 1 stage-0 schedule for the clipped-SGD arms (docs/THEORY_NOTES.md section 1); results/<arm>_<schedule>


In [ ]:
# ---- 1. Pipeline self-test on synthetic caches (about 2 minutes on a GPU): catches environment problems early ----
import subprocess, sys
subprocess.run([sys.executable, 'replicate.py', 'selftest'], check=True)


In [ ]:
# ---- 1b. Condition (S) from the feature caches alone (minutes; no training). Run before anything else: it decides
#          whether the camera-ready delta clause is vacuous on these features (docs/THEORY_NOTES.md section 2) -------
import subprocess, sys, shutil, pathlib
if 'A' in PAPER:
    subprocess.run([sys.executable, 'replicate.py', 'condition-s', '--seeds', SEEDS], check=True)
    (WORK / 'results').mkdir(parents=True, exist_ok=True); shutil.copy('results/paperA_conditionS.json', WORK / 'results' / 'paperA_conditionS.json')


In [ ]:
# ---- 2. Analysis: both optimizer arms, results and tables copied to Drive after each arm -------------------
import subprocess, sys, shutil, pathlib, json
for arm in ARMS:
    for sched in (SCHEDULES if 'A' in PAPER else ['printed']):
        tag = f'{arm}_{sched}' if 'A' in PAPER else arm
        res = pathlib.Path('results') / tag
        subprocess.run([sys.executable, 'replicate.py', '--results', str(res), 'analyze', '--paper', PAPER, '--optimizer', arm,
                        '--seeds', SEEDS, '--ranpac-M', str(RANPAC_M), '--schedule', sched], check=True)
        subprocess.run([sys.executable, 'replicate.py', '--results', str(res), 'figures'], check=True)
        dst = WORK / 'results' / tag; dst.mkdir(parents=True, exist_ok=True)
        for p in res.glob('paper*_results.json'): shutil.copy(p, dst / p.name)
        shutil.copytree(res / 'figures', dst / 'figures', dirs_exist_ok=True)
        shutil.copytree('tables/out', WORK / 'tables_out' / tag, dirs_exist_ok=True)
        print('=' * 30, tag, '=' * 30); print(open('tables/out/DIFF.md').read())
        if 'A' in PAPER:
            A = json.load(open(res / 'paperA_results.json'))
            for k, m in (A.get('method') or {}).items():
                ps = m['per_seed'][0]; print(f"{k}: schedule {ps.get('schedule')} eta {ps.get('eta'):.3g} tau {ps.get('tau'):.3g} clip_frac {ps.get('clip_frac')} max_grad {ps.get('max_grad_norm'):.3g} cond(G) {ps.get('grad_below_half_tau')}")
# Optional: the printed-versus-regenerated ledger, if the TeX sources are on Drive (copy them to WORK/'tex'):
for paper in PAPER:
    tex = WORK / 'tex' / f'paper_{paper}' / 'neurips_2026.tex'
    if tex.exists():
        for arm in ARMS:
            tag = f'{arm}_printed' if 'A' in PAPER else arm
            subprocess.run([sys.executable, 'replicate.py', '--results', str(pathlib.Path('results') / tag), 'audit-tex', '--paper', paper, '--tex', str(tex)], check=True)
            shutil.copy(pathlib.Path('results') / tag / f'TEX_AUDIT_{paper}.md', WORK / 'results' / tag / f'TEX_AUDIT_{paper}.md')


In [ ]:
# ---- 3. Side-by-side: head scale, de-centering outcome and the gauge identity per configuration (Paper B) ----
import json, pathlib
for arm in ARMS:
    p = pathlib.Path('results') / arm / 'paperB_results.json'
    if not p.exists(): continue
    B = json.load(open(p)); print(f'--- {arm}: WT / FIX (Def 7.1) / FIX_c (scale-matched diagnostic) / J / LDA | head scale | per-pair cos | paper-averaged cos')
    for k, r in B['benchmark'].items():
        print(f"{k:28s} {r['WT']['mean']*100:5.1f} {r['FIX']['mean']*100:5.1f} {r['FIX_c']['mean']*100:5.1f} {r['J']['mean']*100:5.1f} {r['LDA']['mean']*100:5.1f} | "
              f"{r['head_scale']['mean']:.2f} | {r['pair_cos_mean']['mean']:.3f} | {r['paper_averaged_cos']['mean']:.4f}")
